# Journey 4 — Prepare SWAN grids and input data

**Learning goals:** Connect a SWAN grid to bathymetry, wind, and wave-boundary data, and understand where data extraction occurs in the workflow.

**Prerequisites:** [Journey 3](journey_03_swan_declarative.ipynb), xarray basics, and the SWAN plugin.


!!! note
    This journey demonstrates configuration and workspace generation. The documentation build does not execute notebook cells or require a SWAN binary. Execution prerequisites are called out separately.


## 1. Define the model grid

The grid is the spatial contract shared by model configuration and data preparation.


In [ ]:
from rompy_swan.grid import SwanGrid

grid = SwanGrid(
    x0=115.0, y0=-32.0, rot=0.0,
    dx=0.25, dy=0.25, nx=9, ny=7,
)
print(grid.bbox())


## 2. Prepare bathymetry

A `SwanDataGrid` describes how a source variable becomes SWAN bottom data. In production, the source may be a local NetCDF file, a catalog, or another registered source. For the render-only path, the important concept is the transformation contract.


In [ ]:
from rompy_swan.data import SwanDataGrid
from rompy.core.source import SourceFile

bottom = SwanDataGrid(
    var="bottom",
    source=SourceFile(uri="path/to/local/bathymetry.nc"),
    z1="elevation",
    fac=-1,
    coords={"x": "lon", "y": "lat"},
)
print(bottom.var, bottom.z1)


## 3. Add wind and wave-boundary data

Wind is represented as a gridded input, while offshore wave information is commonly represented as boundary spectra. Both are connected to the SWAN configuration through interfaces.


In [ ]:
from rompy_swan.data import SwanDataGrid
from rompy_swan.boundary import Boundnest1

wind = SwanDataGrid(
    var="wind",
    source=SourceFile(uri="path/to/local/wind.nc"),
    z1="u10", z2="v10",
    coords={"x": "longitude", "y": "latitude"},
)
boundary = Boundnest1(
    id="offshore",
    source=SourceFile(uri="path/to/local/wave_boundary.nc"),
    sel_method="idw",
    sel_method_kwargs={"tolerance": 4},
)
print(wind.var, boundary.id)


## 4. Connect data through interfaces

The data objects are not standalone output files. They become model inputs when passed to SWAN’s data and boundary interfaces. Data filtering, coordinate mapping, cropping, and interpolation should be inspected before generating the workspace.


In [ ]:
from rompy_swan.interface import DataInterface, BoundaryInterface

inpgrid = DataInterface(bottom=bottom, input=[wind])
boundary_interface = BoundaryInterface(kind=boundary)
print(inpgrid)
print(boundary_interface)


## Checkpoint

The model domain and its three main input families are now explicit. Replace the example URIs with compatible local data when executing this lesson. The documentation build does not fetch or process these files.

**Next:** [Configure SWAN components and outputs](journey_05_swan_components.ipynb).

**Deep dives:** [Boundary nested example](boundary/boundnest1.ipynb) and the [general data-source guide](https://rom-py.github.io/rompy/reference/source/).
